In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

In [ ]:
DATA_DIR = ROOT / "data/complete_dataframe2_30min.csv"
df = pd.read_csv(DATA_DIR,delimiter=",",decimal=".",parse_dates=["ts"],index_col="ts")
df_load = pd.read_csv(ROOT / "data/archive/archive/building_consumption.csv", parse_dates=["timestamp"])

filtered = df_load[
    (df_load["campus_id"] == 1) &
    (df_load["timestamp"] >= "2021-05-14") &
    (df_load["timestamp"] <= "2022-04-11")
]
# How many 15-min readings are there available per meter_id?
counts = filtered.groupby("meter_id").size().reset_index(name="num_readings")

In [ ]:
total_load = filtered.groupby("timestamp")["consumption"].sum().reset_index(name="total_consumption")
total_load["timestamp"] = pd.to_datetime(total_load["timestamp"])
total_load = total_load.set_index("timestamp")

total_load_30min = total_load.resample("30min").mean()

In [ ]:
# Confirm data has 48-step days
total_load_30min["date"] = total_load_30min.index.date
load_counts = total_load_30min.groupby("date").size()
load_complete_days = load_counts[load_counts == 48].index

load_days_df = total_load_30min[total_load_30min["date"].isin(load_complete_days)].copy()

# Make days into 48-step vectors

load_day_profiles = []

for _, day_df in load_days_df.groupby("date"):
    day_profile = day_df.sort_index()["total_consumption"].to_numpy()
    load_day_profiles.append(day_profile)

# Visualization of daily profiles <-- Would the MW scale be per time step or across the day profile?

daily_totals = total_load_30min.groupby(total_load_30min.index.date)["total_consumption"].sum()
print(daily_totals.head())
daily_energy = daily_totals * 0.5
print(daily_energy.describe())

In [ ]:
# Aggregate df across PV modules and wind turbines
df_agg = df.copy()
df_agg["PV_total"] = df_agg[
    ["B117_m", "B319_m", "B330_1_m", "B330_2_m", "B330_3_m", "B716_m", "B715_m"]
].sum(axis=1, min_count=1)

df_agg["Wind_total"] = df_agg[
    ["Aircon_WT Power_m", "Gaia_WT Power_m"]
].sum(axis=1, min_count=1)

df_agg.dropna(subset=["PV_total", "Wind_total"], how="any", inplace=True)

# Keep only complete days
df_agg["date"] = df_agg.index.date
counts_per_day = df_agg.groupby("date").size()
complete_days = counts_per_day[counts_per_day == 48].index
df_agg = df_agg[df_agg["date"].isin(complete_days)].copy()

In [ ]:
# Daily solver
def solve_day_profile(day_df, load_profile, minimum_self_sufficiency, E_capacity, P_capacity, charge_eff, discharge_eff, dT, C_rate, e_price_buy,e_price_sell):
    day_df = day_df.sort_index()

    PV_gen = day_df["PV_total"].to_numpy()
    Wind_gen = day_df["Wind_total"].to_numpy()
    n = len(day_df)

    Pload_np = load_profile

    # Constants
    Pload = cp.Constant(Pload_np)
    Pgen_PV = cp.Constant(PV_gen)
    Pgen_W = cp.Constant(Wind_gen)
    P_capacity = cp.Constant(P_capacity)
    E_capacity = cp.Constant(E_capacity)

    # Variables

    Pcharge = cp.Variable(n, nonneg=True)
    Pdischarge = cp.Variable(n, nonneg=True)
    z = cp.Variable(n, boolean=True)

    Pgrid_buy = cp.Variable(n, nonneg=True)
    Pgrid_sell = cp.Variable(n, nonneg=True)

    SOC = cp.Variable(n)


    constraints = [
        P_capacity <= C_rate * E_capacity,

        Pcharge <= P_capacity * z, #### should it include the mask z or not? 
        Pdischarge <= P_capacity * (1 - z),

        SOC >= 0.1 * E_capacity,
        SOC <= E_capacity,

        # Curtailment can never exceed available generation
        Pgrid_sell <= Pgen_PV + Pgen_W,

        # Power balance 
        Pgen_PV + Pgen_W + Pdischarge + Pgrid_buy == Pload + Pcharge + Pgrid_sell,

        SOC[0] == 0.5 * E_capacity + (Pcharge[0] * charge_eff - Pdischarge[0] / discharge_eff) * dT,
        SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff) * dT,

        SOC[n - 1] >= 0.5 * E_capacity,

        cp.sum(Pgrid_buy) * dT <= (1 - minimum_self_sufficiency) * np.sum(Pload_np) * dT
    ]
    
    # Extras for curtailment -- penalty to avoid spilling and tracker of spillage
    revenue_sold_energy = cp.sum(Pgrid_sell) * dT * e_price_sell

    cost_energy_buy = cp.sum(Pgrid_buy)* dT * e_price_buy # CHECK WHAT UNIT THE ENERGY IS IN kW? WE WANT kW 
    #DKK 

    # SHOULD WE ADD CURTAILMENT OR SELLING ELECTRICITY? 
    problem = cp.Problem(cp.Minimize(cost_energy_buy-revenue_sold_energy), constraints)
    
    try:
        problem.solve(solver=cp.CBC, verbose=False)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            return {
            "status": problem.status,
            "Total_cost_DKK": np.nan,
            "Cost_of_energy_DKK": np.nan,
            "bought_energy_kWh": np.nan,
            "sold_energy_kWh": np.nan,
            "buy_and_sell_simultaneous": np.nan
            }

        both_active = np.any(
            (Pgrid_buy.value > 1e-6) & (Pgrid_sell.value > 1e-6)
        )

        return {
            "status": problem.status,
            "Total_cost_DKK": cost_energy_buy-revenue_sold_energy,
            "Cost_of_energy_DKK": cost_energy_buy,
            "bought_energy_kWh": np.sum(Pgrid_buy.value)*dT,
            "sold_energy_kWh": np.sum(Pgrid_sell.value) * dT,
            "buy_and_sell_simultaneous": both_active
        }

    except Exception as e:
        return {
            "status": f"solver_error: {type(e).__name__}",
            "Total_cost_DKK": np.nan,
            "Cost_of_energy_DKK": np.nan,
            "bought_energy_kWh": np.nan,
            "sold_energy_kWh": np.nan,
            "buy_and_sell_simultaneous": np.nan
        }

In [ ]:
# Settings
dT = 0.5
charge_eff = 0.95
discharge_eff = 0.95
C_rate = 0.5
P_capacity = 2 ###### ADD RESULT FROM OPTIMIZATION 
E_capacity = 2 ####### ADD RESULT FROM OPTIMIZATION 

e_price_buy = 0.349 # EUR/kWh for ROSKILDE
e_price_sell = 0.050 # EUR/kWh for ROSKILDE 

min_self_sufficiency = 0 # NO REQUIREMENT

these temporary electricity prices are according to: https://www.eu-solarcalculator.com/denmark/roskilde

In [ ]:
# How does my generation behave compared to my load?
        # Daily Renewable Generation vs Daily Load

gen_daily = df_agg.groupby("date").apply(
    lambda x: (x["PV_total"].sum() + x["Wind_total"].sum()) * dT
)

load_daily = pd.Series([
    np.sum(profile) * dT for profile in load_day_profiles
])

gen_days = list(df_agg.groupby("date"))


In [ ]:
results = []
for i, (day, day_df) in enumerate(gen_days):
        if len(day_df) != 48:
            continue
        

        load_profile = load_day_profiles[i % len(load_day_profiles)]

        out = solve_day_profile(day_df,load_profile,min_self_sufficiency,E_capacity,P_capacity,charge_eff,discharge_eff,dT,C_rate,e_price_buy,e_price_sell)
        out["date"] = pd.to_datetime(day)
        results.append(out)

results_df = pd.DataFrame(results)

print(results_df)
print(results_df["status"].value_counts(dropna=False))
print(results_df["buy_and_sell_simultaneous"].value_counts())